In [2]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


Authenticated using refresh token.


In [4]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [111.80, -7.40],
            [112.10, -7.40],
            [112.10, -7.70],
            [111.80, -7.70],
            [111.80, -7.40],
        ]
    ]
}

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-25", "2026-08-25"],
    spatial_extent={
        "west": 111.80,
        "south": -7.70,
        "east": 112.10,
        "north": -7.40
    },
    bands=["NO2"],
)

# Agregasi harian agar tidak ada lebih dari satu data per hari
s5p_no2_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial untuk menghasilkan rata-rata time series per AOI
s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer="mean", geometries=aoi)

In [5]:
job = s5post.execute_batch(title="NO2 di Nganjuk", outputfile="NO2DiNganjuk.nc")

0:00:00 Job 'j-260827091200442abf9b2bc55425e4e1': send 'start'
0:00:02 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:00:08 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:00:15 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:00:23 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:00:33 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:00:45 Job 'j-260827091200442abf9b2bc55425e4e1': queued (progress 0%)
0:01:01 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:01:20 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:01:44 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:02:15 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:02:52 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:03:39 Job 'j-260827091200442abf9b2bc55425e4e1': running (progress N/A)
0:04:38 Job 'j-260827091200442abf9b2bc55425e4e1': finished (progress 100%

In [11]:
import numpy as np
import pandas as pd
import netCDF4

file_path = "CODiNganjuk.nc"
ds = netCDF4.Dataset(file_path)
# Ambil NO2
no2 = ds.variables["CO"][:]

# Ambil Time
time = ds.variables["t"][:]

# Konversi waktu ke format tanggal
try:
    time_units = ds.variables["t"].units
    dates = netCDF4.num2date(time, units=time_units)
except Exception:
    dates = time  # fallback jika tidak ada units

no2_filled = np.zeros_like(no2)
no2_filled = no2_filled.filled(0)

# Loop tiap grid (y, x)
for i in range(no2.shape[1]):     # 9 baris
    for j in range(no2.shape[2]): # 8 kolom
        series = pd.Series(no2[:, i, j])
        no2_filled[:, i, j] = series.interpolate(
            method='linear', limit_direction='both'
        ).to_numpy()
        
new_dates = []
new_no2 = []

for i in range(len(dates)):
    new_date = dates[i].strftime('%Y-%m-%d')
    new_dates.append(new_date)
    new_no2.append(np.mean(no2_filled[i]))

df = pd.DataFrame({
    "date": new_dates,
    "CO": new_no2
})

# Simpan ke CSV
df.to_csv("CO_Nganjuk_timeseries.csv", index=False)